# Mean Variance Optimization with Scipy

In [ ]:
import sys
sys.path.insert(0,'C:\\code\\python_for_the_financial_economist\\')

# import relevant packages
import numpy as np
import pandas as pd
from scipy import stats, optimize
import matplotlib.pyplot as plt

# own functions from codelib
from codelib.statistics import moments as mom
from codelib.portfolio_optimization.mean_variance import portfolio_mean, portfolio_std, portfolio_variance, minimum_variance_portfolio

from codelib.visualization.layout import DefaultStyle
DefaultStyle();

Consider the portfolio optimization problem 


$$
\arg \min \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w} \; \; \text{st. } \mathbf{1}^\top \mathbf{w} = 1
$$

where the expected return vector is given by 

$$
\boldsymbol{\mu} = \begin{pmatrix} 0.032 \\ 0.0322 \\ 0.084 \\ 0.082 \end{pmatrix},
$$

the vector of volatilites is given by 

$$
\mathbf{v} = \begin{pmatrix} 0.05 \\ 0.05 \\ 0.22 \\ 0.22 \end{pmatrix},
$$

and the correlation matrix is given by 

$$
\mathbf{C} = \begin{pmatrix} 1.0 & 0.85 & 0.5 & 0.45 \\
                      0.85 & 1.0 & 0.5 & 0.45 \\
                      0.5 & 0.5 & 1.0 & 0.9 \\
                      0.45 & 0.45 & 0.9 & 1.0 \end{pmatrix}
$$

In [ ]:
corr_mat = np.array([[1.0, 0.85, 0.5, 0.45],
                     [0.85, 1.0, 0.5, 0.45],
                     [0.5, 0.5, 1.0, 0.9],
                     [0.45, 0.45, 0.9, 1.0]])

vols = np.array([5.0, 5.0, 22.0, 22.0]) / 100.0
mu = np.array([3.2, 3.22, 8.4, 8.2]) / 100.0

cov_mat = mom.corr_to_cov_matrix(corr_mat, vols)

## Problem 1

Set up the minimization problem that enable you to find the minimum variance portfolio using `scipy.optimize`. Apply a budget constraint requiring the portfolio weights to sum to one. Compare with the analytical solution. 

### Solution

We have the minimization problem 

$$
\mathbf{w}^{Min-Var} = \arg \min \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w} \; \; \text{st. } \mathbf{1}^\top \mathbf{w} = 1
$$

The analytical solution is given by 

$$
\mathbf{w}^{Min-Var} = \frac{\boldsymbol{\Sigma}^{-1} \mathbf{1}}{\mathbf{1}^\top \boldsymbol{\Sigma}^{-1} \mathbf{1}}
$$

In [ ]:
"""
Solving with scipy
"""

# define sum to one constraint
sum_to_one_cons = {'type': 'eq',
                   'fun' : lambda x: np.sum(x) - 1.0, 
                   'jac' : lambda x: np.ones_like(x)} 

# define initial guess 
x0 = np.ones_like(mu) / len(mu)

# define Jacobian 
port_var_der = lambda w, cov_mat: 2 * w @ cov_mat


res = optimize.minimize(portfolio_variance, x0=x0, args=(cov_mat,),
                        method='SLSQP',
                        jac=port_var_der,
                        constraints=[sum_to_one_cons],  # no_short_cons,
                        options={'ftol': 1e-9, 'disp': True}) #, bounds=bounds)

w_min_var_opt = res.x

# show results
res

In [ ]:
"""
Solving with scipy - not providing gradient information
"""

# define sum to one constraint
sum_to_one_cons = {'type': 'eq',
                   'fun' : lambda x: np.sum(x) - 1.0} 

# define initial guess 
x0 = np.ones_like(mu) / len(mu)

res = optimize.minimize(portfolio_variance, x0=x0, args=(cov_mat,),
                        method='SLSQP',
                        constraints=[sum_to_one_cons],  # no_short_cons,
                        options={'ftol': 1e-9, 'disp': True}) #, bounds=bounds)

w_min_var_opt = res.x

# show results
res

In [ ]:
print("w_min_var_opt:", w_min_var_opt)

In [ ]:
minimum_variance_portfolio(cov_mat)

## Problem 2

Minimize the portfolio variance subject to the constraints that 

* The expected return should be above 6%
* No shorting (all weights should be equal or greater than 0)
* The portfolio weights need to sum to one (budget constraint)

### Solution

In [ ]:
# budget constraint 
sum_to_one_cons = {'type': 'eq',
                   'fun' : lambda x: np.sum(x) - 1.0, 
                   'jac' : lambda x: np.ones_like(x)}

# no shorting constraint
no_short_cons = {'type': 'ineq',
                 'fun' : lambda x: x,
                 'jac' : lambda x: np.eye(len(x))}

# alternatively use 
bounds = [(0.0, 1.0)] * len(mu)

# define return constraint 
target_cons = {'type': 'ineq',
               'fun' : lambda x: x @ mu - 0.06,
               'jac' : lambda x: mu}

port_var_der = lambda w, cov_mat: 2 * w @ cov_mat


x0 = np.ones_like(mu) / len(mu)
res = optimize.minimize(portfolio_variance, x0=x0, args=(cov_mat,),
                        method='SLSQP',
                        jac=port_var_der,
                        constraints=[sum_to_one_cons,
                                     no_short_cons, 
                                     target_cons],  # no_short_cons,
                        options={'ftol': 1e-9, 'disp': True}) #, bounds=bounds)

w_mean_var_opt = res.x

res

In [ ]:
w_mean_var_opt